# 📊 Notebook 9 — Cálculo de Indicadores Financeiros (Camada Gold)

## Contexto

Este notebook calcula todos os indicadores definidos em `indicadores_financeiros.md`,
usando exclusivamente as variáveis mapeadas em `premissas_calculo.md`.

**Engenharia reversa garantida:** cada coluna do mart possui prefixo que identifica sua origem:
- `V__`   → variável de input (mapeamento direto a um `CD_CONTA` da Silver)
- `AUX__` → variável auxiliar (combinação de variáveis V, sem interpretação financeira direta)
- `IND__` → indicador financeiro final (fórmula auditável em `indicadores_financeiros.md`)

**Saída:** `layer_03_gold.mart_indicadores_financeiros`
> Uma linha por `(CNPJ_CIA, DT_REFER)` com todas as dimensões, inputs, auxiliares e indicadores.

**Referências:**
- `study_guides/indicadores/premissas_calculo.md`
- `study_guides/indicadores/indicadores_financeiros.md`


## 1 — Setup

In [1]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv()

def create_db_engine():
    user     = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host     = os.getenv('DB_HOST', 'localhost')
    port     = os.getenv('DB_PORT', '5432')
    dbname   = os.getenv('DB_NAME', 'data_lake')
    return create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

engine = create_db_engine()

# Tabelas Silver
BP  = 'layer_02_silver.n1_dfp_cia_aberta_bp'
DRE = 'layer_02_silver.n1_dfp_cia_aberta_dre'
DFC = 'layer_02_silver.n1_dfp_cia_aberta_dfc'
CAD = 'layer_01_bronze.cad_cia_aberta'

# Tabela Gold de destino
GOLD_SCHEMA = 'layer_03_gold'
GOLD_TABLE  = 'mart_indicadores_financeiros'

print(f'✅ Engine criado.')
print(f'   BP  → {BP}')
print(f'   DRE → {DRE}')
print(f'   DFC → {DFC}')
print(f'   OUT → {GOLD_SCHEMA}.{GOLD_TABLE}')


✅ Engine criado.
   BP  → layer_02_silver.n1_dfp_cia_aberta_bp
   DRE → layer_02_silver.n1_dfp_cia_aberta_dre
   DFC → layer_02_silver.n1_dfp_cia_aberta_dfc
   OUT → layer_03_gold.mart_indicadores_financeiros


## 2 — Helper de Divisão Segura

In [2]:
def safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    """
    Divisão segura: retorna NaN quando denominador = 0 ou NaN.
    Evita ZeroDivisionError e infinity nos indicadores.
    """
    d = den.astype(float).replace(0, np.nan)
    return num.astype(float) / d

print('✅ safe_div definido.')


✅ safe_div definido.


## 3 — Extração BP (Pivot Wide)

Contas extraídas por premissa:

| Alias Python | CD_CONTA  | Variável | Descrição |
|---|---|---|---|
| `v01_ativo_total` | `1` | V01 | Ativo Total |
| `v02_ativo_circ` | `1.01` | V02 | Ativo Circulante |
| `v03_ativo_ncirc` | `1.02` | V03 | Ativo Não Circulante |
| `v04_rlp` | `1.02.01` | V04 | Realizável a Longo Prazo |
| `v05_imobilizado` | `1.02.03` | V05 | Ativo Imobilizado |
| `v06_estoques_raw` | `1.01.04` | V06 | Estoques (raw — COALESCE aplicado depois) |
| `v07_contas_receber` | `1.01.03` | V07 | Contas a Receber CP |
| `v09_fornecedores` | `2.01.02` | V09 | Fornecedores |
| `v10_passivo_circ` | `2.01` | V10 | Passivo Circulante |
| `v11_passivo_ncirc` | `2.02` | V11 | Passivo Não Circulante (ELP) |
| `v12_pl` | `2.03` | V12 | Patrimônio Líquido |
| `v13_emp_cp` | `2.01.04` | V13/V16 | Empréstimos CP (também = PCF Fleuriet) |
| `v14_emp_lp` | `2.02.01` | V14 | Empréstimos LP |
| `v22_caixa_bp` | `1.01.01` | V22 | Caixa e Equivalentes (BP) |
| `v15b_aplic_fin` | `1.01.02` | V15(parte) | Aplicações Financeiras (parte do ACF Fleuriet) |

> **Deduplicação:** para cada `(CNPJ_CIA, DT_REFER, CD_CONTA)` prefere `FLAG_RECONSTRUCAO = False`
> (dado original auditado). Só usa reconstruído se o original não existir.


In [3]:
Q_BP = f"""
WITH ranked AS (
    SELECT
        "CNPJ_CIA", "DT_REFER", "CD_CONTA", "VL_CONTA_TRATADO",
        ROW_NUMBER() OVER (
            PARTITION BY "CNPJ_CIA", "DT_REFER", "CD_CONTA"
            ORDER BY "FLAG_RECONSTRUCAO" ASC   -- False(0) antes de True(1): prefere original
        ) AS rn
    FROM {BP}
    WHERE "CD_CONTA" IN (
        '1', '1.01', '1.02', '1.02.01', '1.02.03',
        '1.01.04', '1.01.03', '1.01.01', '1.01.02',
        '2.01', '2.02', '2.03', '2.01.02', '2.01.04', '2.02.01'
    )
),
deduped AS (
    SELECT "CNPJ_CIA", "DT_REFER", "CD_CONTA", "VL_CONTA_TRATADO"
    FROM ranked WHERE rn = 1
)
SELECT
    "CNPJ_CIA",
    "DT_REFER",
    MAX(CASE WHEN "CD_CONTA" = '1'       THEN "VL_CONTA_TRATADO" END) AS v01_ativo_total,
    MAX(CASE WHEN "CD_CONTA" = '1.01'    THEN "VL_CONTA_TRATADO" END) AS v02_ativo_circ,
    MAX(CASE WHEN "CD_CONTA" = '1.02'    THEN "VL_CONTA_TRATADO" END) AS v03_ativo_ncirc,
    MAX(CASE WHEN "CD_CONTA" = '1.02.01' THEN "VL_CONTA_TRATADO" END) AS v04_rlp,
    MAX(CASE WHEN "CD_CONTA" = '1.02.03' THEN "VL_CONTA_TRATADO" END) AS v05_imobilizado,
    MAX(CASE WHEN "CD_CONTA" = '1.01.04' THEN "VL_CONTA_TRATADO" END) AS v06_estoques_raw,
    MAX(CASE WHEN "CD_CONTA" = '1.01.03' THEN "VL_CONTA_TRATADO" END) AS v07_contas_receber,
    MAX(CASE WHEN "CD_CONTA" = '1.01.01' THEN "VL_CONTA_TRATADO" END) AS v22_caixa_bp,
    MAX(CASE WHEN "CD_CONTA" = '1.01.02' THEN "VL_CONTA_TRATADO" END) AS v15b_aplic_fin,
    MAX(CASE WHEN "CD_CONTA" = '2.01'    THEN "VL_CONTA_TRATADO" END) AS v10_passivo_circ,
    MAX(CASE WHEN "CD_CONTA" = '2.02'    THEN "VL_CONTA_TRATADO" END) AS v11_passivo_ncirc,
    MAX(CASE WHEN "CD_CONTA" = '2.03'    THEN "VL_CONTA_TRATADO" END) AS v12_pl,
    MAX(CASE WHEN "CD_CONTA" = '2.01.02' THEN "VL_CONTA_TRATADO" END) AS v09_fornecedores,
    MAX(CASE WHEN "CD_CONTA" = '2.01.04' THEN "VL_CONTA_TRATADO" END) AS v13_emp_cp,
    MAX(CASE WHEN "CD_CONTA" = '2.02.01' THEN "VL_CONTA_TRATADO" END) AS v14_emp_lp
FROM deduped
GROUP BY "CNPJ_CIA", "DT_REFER"
ORDER BY "CNPJ_CIA", "DT_REFER";
"""

with engine.connect() as conn:
    df_bp = pd.read_sql(text(Q_BP), conn)

print(f'✅ BP extraído: {len(df_bp):,} linhas  |  {df_bp["CNPJ_CIA"].nunique()} empresas  |  {df_bp["DT_REFER"].nunique()} períodos')
df_bp.head(3)


✅ BP extraído: 2,220 linhas  |  185 empresas  |  54 períodos


,CNPJ_CIA,DT_REFER,v01_ativo_total,v02_ativo_circ,v03_ativo_ncirc,v04_rlp,v05_imobilizado,v06_estoques_raw,v07_contas_receber,v22_caixa_bp,v15b_aplic_fin,v10_passivo_circ,v11_passivo_ncirc,v12_pl,v09_fornecedores,v13_emp_cp,v14_emp_lp
0,00.070.698/0001-11,2010-12-31,2.119934e+09,449960000.0,1.669974e+09,799490000.0,197361000.0,8619000.0,321170000.0,99258000.0,NaN,627946000.0,777682000.0,714306000.0,142987000.0,154199000.0,332030000.0
1,00.070.698/0001-11,2011-12-31,2.170285e+09,457284000.0,1.713001e+09,820292000.0,193114000.0,9108000.0,357186000.0,66748000.0,NaN,657765000.0,766489000.0,746031000.0,155447000.0,127599000.0,297884000.0
2,00.070.698/0001-11,2012-12-31,2.424419e+09,570535000.0,1.853884e+09,746751000.0,198201000.0,8540000.0,308111000.0,185433000.0,9805000.0,615292000.0,987141000.0,821986000.0,168579000.0,106013000.0,315813000.0


## 4 — Extração DRE (Pivot Wide)

| Alias Python | CD_CONTA | Variável | Descrição |
|---|---|---|---|
| `v17_receita_liq` | `3.01` | V17 | Receita Líquida de Vendas |
| `v18_cpv` | `3.02` | V18 | CPV (⚠️ valor **negativo** na Silver) |
| `v19_lucro_bruto` | `3.03` | V19 | Lucro Bruto |
| `v20_ebit` | `3.05` | V20 | EBIT / Lucro Operacional |
| `v21_lucro_liq` | `3.11` | V21 | Lucro Líquido |
| `v25_lpa_basico` | `3.99.01.01` | V25 | LPA Básico ON (outlier filtrado: ABS > 10.000) |
| `v26_lpa_diluido_raw` | `3.99.02.01` | V26(raw) | LPA Diluído ON (COALESCE(V26, V25) aplicado depois) |

> **Outlier LPA:** Marisa Lojas 2021 tem `3.99.01.01 = -27.4 quadrilhões` — erro na fonte CVM.
> Filtro: `ABS(VL_CONTA_TRATADO) > 10.000` → NULL antes do pivot.


In [4]:
Q_DRE = f"""
WITH ranked AS (
    SELECT
        "CNPJ_CIA", "DT_REFER", "CD_CONTA", "VL_CONTA_TRATADO",
        ROW_NUMBER() OVER (
            PARTITION BY "CNPJ_CIA", "DT_REFER", "CD_CONTA"
            ORDER BY "FLAG_RECONSTRUCAO" ASC
        ) AS rn
    FROM {DRE}
    WHERE "CD_CONTA" IN ('3.01','3.02','3.03','3.05','3.11','3.99.01.01','3.99.02.01')
),
deduped AS (
    SELECT "CNPJ_CIA", "DT_REFER", "CD_CONTA",
           -- Premissa N3: filtrar outlier LPA (Marisa Lojas 2021 = -27.4 quadrilhões)
           CASE
               WHEN "CD_CONTA" IN ('3.99.01.01','3.99.02.01')
                AND ABS("VL_CONTA_TRATADO") > 10000
               THEN NULL
               ELSE "VL_CONTA_TRATADO"
           END AS "VL_CONTA_TRATADO"
    FROM ranked WHERE rn = 1
)
SELECT
    "CNPJ_CIA",
    "DT_REFER",
    MAX(CASE WHEN "CD_CONTA" = '3.01'       THEN "VL_CONTA_TRATADO" END) AS v17_receita_liq,
    MAX(CASE WHEN "CD_CONTA" = '3.02'       THEN "VL_CONTA_TRATADO" END) AS v18_cpv,
    MAX(CASE WHEN "CD_CONTA" = '3.03'       THEN "VL_CONTA_TRATADO" END) AS v19_lucro_bruto,
    MAX(CASE WHEN "CD_CONTA" = '3.05'       THEN "VL_CONTA_TRATADO" END) AS v20_ebit,
    MAX(CASE WHEN "CD_CONTA" = '3.11'       THEN "VL_CONTA_TRATADO" END) AS v21_lucro_liq,
    MAX(CASE WHEN "CD_CONTA" = '3.99.01.01' THEN "VL_CONTA_TRATADO" END) AS v25_lpa_basico,
    MAX(CASE WHEN "CD_CONTA" = '3.99.02.01' THEN "VL_CONTA_TRATADO" END) AS v26_lpa_diluido_raw
FROM deduped
GROUP BY "CNPJ_CIA", "DT_REFER"
ORDER BY "CNPJ_CIA", "DT_REFER";
"""

with engine.connect() as conn:
    df_dre = pd.read_sql(text(Q_DRE), conn)

print(f'✅ DRE extraído: {len(df_dre):,} linhas  |  {df_dre["CNPJ_CIA"].nunique()} empresas')
df_dre.head(3)


✅ DRE extraído: 2,221 linhas  |  185 empresas


,CNPJ_CIA,DT_REFER,v17_receita_liq,v18_cpv,v19_lucro_bruto,v20_ebit,v21_lucro_liq,v25_lpa_basico,v26_lpa_diluido_raw
0,00.070.698/0001-11,2010-12-31,1.839320e+09,-6.433260e+08,1.195994e+09,83684000.0,19617000.0,NaN,NaN
1,00.070.698/0001-11,2011-12-31,1.377619e+09,-1.069722e+09,3.078970e+08,144133000.0,46132000.0,2.85201,NaN
2,00.070.698/0001-11,2012-12-31,1.628678e+09,-1.303059e+09,3.256190e+08,55482000.0,74679000.0,6.04369,6.04369


## 5 — Extração DFC (Pivot Wide)

| Alias Python | CD_CONTA | Variável | Descrição |
|---|---|---|---|
| `v23_caixa_dfc` | `6.05.02` | V23 | Saldo Final de Caixa e Equivalentes (DFC) |

> **Premissa N1:** usar `6.05.02` (DFC) para Dívida Líquida e EV — inclui CDBs/LFTs ≤90d (CPC 03)
> e caixa de grupos a venda (CPC 31). **Não usar** `1.01.01` (BP) para esses fins.


In [5]:
Q_DFC = f"""
SELECT
    "CNPJ_CIA",
    "DT_REFER",
    MAX("VL_CONTA_TRATADO") AS v23_caixa_dfc
FROM {DFC}
WHERE "CD_CONTA" = '6.05.02'
GROUP BY "CNPJ_CIA", "DT_REFER"
ORDER BY "CNPJ_CIA", "DT_REFER";
"""

with engine.connect() as conn:
    df_dfc = pd.read_sql(text(Q_DFC), conn)

print(f'✅ DFC extraído: {len(df_dfc):,} linhas  |  {df_dfc["CNPJ_CIA"].nunique()} empresas')
df_dfc.head(3)


✅ DFC extraído: 2,179 linhas  |  182 empresas


,CNPJ_CIA,DT_REFER,v23_caixa_dfc
0,00.336.701/0001-04,2014-12-31,249074000.0
1,00.336.701/0001-04,2015-12-31,256782000.0
2,00.336.701/0001-04,2016-12-31,282735000.0


## 6 — Dimensões (cad_cia_aberta)

Colunas descritivas para identificação no dashboard:
- `DENOM_CIA` → razão social
- `SETOR_ATIV` → setor de atividade
- `TP_MERC`   → tipo de mercado (= `'BOLSA'` para todas, por filtro do pipeline)

> **Filtro:** mesmo universo da camada Silver — `SIT = 'ATIVO'`, `TP_MERC = 'BOLSA'`,
> `SIT_EMISSOR = 'FASE OPERACIONAL'`.


In [6]:
Q_DIM = f"""
SELECT
    "CNPJ_CIA",
    "DENOM_SOCIAL"  AS razao_social,
    "SETOR_ATIV" AS setor,
    "TP_MERC"    AS tp_merc
FROM {CAD}
WHERE "SIT"          = 'ATIVO'
  AND "TP_MERC"      = 'BOLSA'
  AND "SIT_EMISSOR"  = 'FASE OPERACIONAL';
"""

with engine.connect() as conn:
    df_dim = pd.read_sql(text(Q_DIM), conn)

print(f'✅ Dimensões extraídas: {len(df_dim):,} empresas')
df_dim.head(3)


✅ Dimensões extraídas: 307 empresas


,CNPJ_CIA,razao_social,setor,tp_merc
0,12.528.708/0001-07,AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. ...,"Máquinas, Equipamentos, Veículos e Peças",BOLSA
1,10.338.320/0001-00,AFLUENTE TRANSMISSÃO DE ENERGIA ELETRICA S/A,Energia Elétrica,BOLSA
2,17.167.396/0001-69,ALFA HOLDINGS S.A.,Emp. Adm. Part. - Intermediação Financeira,BOLSA


## 7 — Montagem da Tabela Wide + COALESCE das Variáveis de Input

**Joins:**
- `BP OUTER JOIN DRE` → empresas com BP mas sem DRE (e vice-versa) ficam como NaN
- `LEFT JOIN DFC` → DFC pode ser menor (182 vs 185 empresas)
- `LEFT JOIN dim` → dimensões descritivas

**COALESCE aplicados nesta etapa (conforme premissas_calculo.md):**

| Variável | Regra | Justificativa |
|---|---|---|
| `v06_estoques` | `COALESCE(raw, 0)` | Empresas de serviço puro sem estoque — tratar como zero |
| `v13_emp_cp` | `COALESCE(raw, 0)` | Empresa sem dívida CP — ausência = zero |
| `v14_emp_lp` | `COALESCE(raw, 0)` | Empresa sem dívida LP — ausência = zero |
| `v15b_aplic_fin` | `COALESCE(raw, 0)` | Sem aplicações financeiras = zero no ACF |
| `v26_lpa_diluido` | `COALESCE(v26, v25)` | 29 empresas sem diluído; gap temporal pré-2015 |

> `v06_estoques_raw` é mantido separadamente para que indicadores que devem ser N/A
> (Giro Estoques, PMRE, Ciclo Econômico) exibam NULL quando empresa não tem estoque.


In [7]:
print('🔗 Fazendo merge das 4 fontes...')

df = (
    df_bp
    .merge(df_dre, on=['CNPJ_CIA', 'DT_REFER'], how='outer')
    .merge(df_dfc, on=['CNPJ_CIA', 'DT_REFER'], how='left')
    .merge(df_dim, on='CNPJ_CIA',               how='left')
)

print(f'   Linhas após merge: {len(df):,}')
print(f'   Empresas:          {df["CNPJ_CIA"].nunique()}')
print(f'   Períodos:          {df["DT_REFER"].nunique()}')

# ── COALESCE das variáveis de input ──────────────────────────────────────────
df['v06_estoques']    = df['v06_estoques_raw'].fillna(0)          # V06: zero = empresa sem estoque
df['v13_emp_cp']      = df['v13_emp_cp'].fillna(0)                # V13: zero = sem dívida CP
df['v14_emp_lp']      = df['v14_emp_lp'].fillna(0)                # V14: zero = sem dívida LP
df['v15b_aplic_fin']  = df['v15b_aplic_fin'].fillna(0)            # V15b: zero = sem aplicações
df['v26_lpa_diluido'] = df['v26_lpa_diluido_raw'].fillna(         # V26: COALESCE(diluído, básico)
    df['v25_lpa_basico']
)

print('\n✅ COALESCEs aplicados:')
print(f'   v06 NULLs restantes (sem estoque estrutural): {df["v06_estoques_raw"].isna().sum()}')
print(f'   v26 cobertura após COALESCE: {df["v26_lpa_diluido"].notna().sum()} / {len(df)} ({df["v26_lpa_diluido"].notna().mean()*100:.1f}%)')


🔗 Fazendo merge das 4 fontes...
   Linhas após merge: 2,221
   Empresas:          185
   Períodos:          54

✅ COALESCEs aplicados:
   v06 NULLs restantes (sem estoque estrutural): 265
   v26 cobertura após COALESCE: 1925 / 2221 (86.7%)


## 8 — Variáveis Auxiliares (prefixo `AUX_`)

Combinações intermediárias reutilizadas em múltiplos indicadores:

| Coluna | Fórmula | Usada em |
|---|---|---|
| `AUX_passivo_total` | V10 + V11 | PCT/CP, PCT/AT, Garantia, Composição |
| `AUX_acf` | V22 + V15b | Saldo de Tesouraria (Fleuriet) |
| `AUX_divida_bruta` | V13 + V14 | Dívida Líquida, ROI |
| `AUX_divida_liquida` | AUX_divida_bruta − V23 | Dívida Líquida (indicador futuro EV) |
| `AUX_capital_invest` | V12 + V13 + V14 | ROI (denominador = PL + Passivo Oneroso) |
| `AUX_cpv_abs` | ABS(V18) | Giro Estoques, Giro CP, PMRE, PMPC (V18 é negativo na Silver) |


In [8]:
print('⚙️  Calculando auxiliares...')

df['AUX_passivo_total']  = df['v10_passivo_circ'] + df['v11_passivo_ncirc']
df['AUX_acf']            = df['v22_caixa_bp']     + df['v15b_aplic_fin']      # ACF Fleuriet = V15
df['AUX_divida_bruta']   = df['v13_emp_cp']        + df['v14_emp_lp']
df['AUX_divida_liquida'] = df['AUX_divida_bruta']  - df['v23_caixa_dfc']
df['AUX_capital_invest'] = df['v12_pl'] + df['v13_emp_cp'] + df['v14_emp_lp']
df['AUX_cpv_abs']        = df['v18_cpv'].abs()      # V18 vem negativo da CVM

print('✅ Auxiliares calculados:')
for col in [c for c in df.columns if c.startswith('AUX_')]:
    n_null = df[col].isna().sum()
    print(f'   {col}: {n_null} NULLs')


⚙️  Calculando auxiliares...
✅ Auxiliares calculados:
   AUX_passivo_total: 3 NULLs
   AUX_acf: 4 NULLs
   AUX_divida_bruta: 0 NULLs
   AUX_divida_liquida: 42 NULLs
   AUX_capital_invest: 1 NULLs
   AUX_cpv_abs: 29 NULLs


## 9 — Indicadores Financeiros (prefixo `IND_`)

### 9.1 Liquidez (Seção 1 de `indicadores_financeiros.md`)


In [9]:
print('💧 Calculando indicadores de Liquidez...')

# 1.1 Liquidez Geral = (AC + RLP) / (PC + ELP)
df['IND_liquidez_geral']    = safe_div(
    df['v02_ativo_circ'] + df['v04_rlp'],
    df['AUX_passivo_total']
)

# 1.2 Liquidez Corrente = AC / PC
df['IND_liquidez_corrente'] = safe_div(df['v02_ativo_circ'],  df['v10_passivo_circ'])

# 1.3 Liquidez Seca = (AC - Estoques) / PC
# v06_estoques = COALESCE(raw, 0): quando empresa não tem estoque, LS = LC (correto)
df['IND_liquidez_seca']     = safe_div(
    df['v02_ativo_circ'] - df['v06_estoques'],
    df['v10_passivo_circ']
)

# 1.4 Liquidez Imediata = Disponibilidades / PC
# Premissa N1: usar 1.01.01 (BP), não 6.05.02 (DFC), pois este ratio é definido sobre o balanço
df['IND_liquidez_imediata'] = safe_div(df['v22_caixa_bp'], df['v10_passivo_circ'])

print('✅ Liquidez OK')
df[['CNPJ_CIA','DT_REFER','IND_liquidez_geral','IND_liquidez_corrente',
    'IND_liquidez_seca','IND_liquidez_imediata']].dropna().head(3)


💧 Calculando indicadores de Liquidez...
✅ Liquidez OK


,CNPJ_CIA,DT_REFER,IND_liquidez_geral,IND_liquidez_corrente,IND_liquidez_seca,IND_liquidez_imediata
0,00.070.698/0001-11,2010-12-31,0.888891,0.716558,0.702833,0.158068
1,00.070.698/0001-11,2011-12-31,0.897014,0.695209,0.681362,0.101477
2,00.070.698/0001-11,2012-12-31,0.822054,0.927259,0.913379,0.301374


### 9.2 Endividamento e Estrutura de Capital (Seção 2)

In [10]:
print('🏦 Calculando indicadores de Endividamento...')

# 2.1 PCT/CP = (PC + ELP) / PL
df['IND_pct_cp']           = safe_div(df['AUX_passivo_total'], df['v12_pl'])

# 2.2 PCT/AT = (PC + ELP) / AT
df['IND_pct_at']           = safe_div(df['AUX_passivo_total'], df['v01_ativo_total'])

# 2.3 Garantia CP/CT = PL / (PC + ELP)
df['IND_garantia_ct']      = safe_div(df['v12_pl'],            df['AUX_passivo_total'])

# 2.4 Composição de Endividamento = PC / (PC + ELP)
df['IND_composicao_endiv'] = safe_div(df['v10_passivo_circ'],  df['AUX_passivo_total'])

# 2.5 Imobilização CP = Imobilizado / PL
df['IND_imob_cp']          = safe_div(df['v05_imobilizado'],   df['v12_pl'])

# 2.6 Imobilização AT = Imobilizado / AT
df['IND_imob_at']          = safe_div(df['v05_imobilizado'],   df['v01_ativo_total'])

print('✅ Endividamento OK')


🏦 Calculando indicadores de Endividamento...
✅ Endividamento OK


### 9.3 Margem de Lucro e LPA (Seção 3)

In [11]:
print('📈 Calculando indicadores de Margem...')

# 3.1 Margem Bruta = Lucro Bruto / Receita Líquida
df['IND_margem_bruta']       = safe_div(df['v19_lucro_bruto'], df['v17_receita_liq'])

# 3.2 Margem Operacional = EBIT / Receita Líquida
# Premissa V20: EBIT = 3.05 (ST_CONTA_FIXA='S'). NÃO usar 3.07 (já inclui resultado financeiro)
df['IND_margem_operacional'] = safe_div(df['v20_ebit'],        df['v17_receita_liq'])

# 3.3 Margem Líquida = Lucro Líquido / Receita Líquida
df['IND_margem_liquida']     = safe_div(df['v21_lucro_liq'],   df['v17_receita_liq'])

# 3.4 LPA Diluído — já calculado via COALESCE(V26, V25) na etapa de merge
# Premissa N3: NÃO usar 3.99 (pai) nem 3.99.02 (pai) — usar leaf 3.99.02.01
# Outlier Marisa 2021 já removido no SQL (ABS > 10.000)
df['IND_lpa_diluido']        = df['v26_lpa_diluido']

print('✅ Margem OK')


📈 Calculando indicadores de Margem...
✅ Margem OK


### 9.4 Rentabilidade (Seção 4)

In [12]:
print('💰 Calculando indicadores de Rentabilidade...')

# 4.1 ROA = Lucro Líquido / Ativo Total
df['IND_roa'] = safe_div(df['v21_lucro_liq'], df['v01_ativo_total'])

# 4.2 ROE = Lucro Líquido / PL
# Atenção: PL pode ser negativo (empresas com prejuízos acumulados).
# ROE negativo + PL negativo = ROE positivo aparente → tratar no dashboard com flag
df['IND_roe'] = safe_div(df['v21_lucro_liq'], df['v12_pl'])

# 4.3 ROI = Lucro Líquido / (PL + Passivo Oneroso)
# Passivo Oneroso = V13 (Emp CP) + V14 (Emp LP) → AUX_capital_invest
df['IND_roi'] = safe_div(df['v21_lucro_liq'], df['AUX_capital_invest'])

print('✅ Rentabilidade OK')


💰 Calculando indicadores de Rentabilidade...
✅ Rentabilidade OK


### 9.5 Atividade — Giros e Prazos Médios (Seção 5)

> **AUX_cpv_abs** = ABS(V18): CPV vem negativo na Silver.
> Giro de Estoques e PMRE usam `v06_estoques_raw` (não o COALESCE com zero):
> quando `v06_estoques_raw = NULL` (empresa sem estoque), o indicador retorna NULL (N/A),
> conforme premissa N2 de `premissas_calculo.md`.


In [13]:
print('🔄 Calculando indicadores de Atividade...')

# 5.1 Giro Estoques = CPV / Estoques
# NULL quando empresa não tem estoque (v06_estoques_raw = NULL) — N/A intencional
df['IND_giro_estoques'] = safe_div(df['AUX_cpv_abs'], df['v06_estoques_raw'])

# 5.2 Giro Contas a Receber = Receita / Clientes CP
# Premissa V08: clientes LP desconsiderados (ST_CONTA_FIXA='N', cobertura baixa)
df['IND_giro_cr']       = safe_div(df['v17_receita_liq'],  df['v07_contas_receber'])

# 5.3 Giro Contas a Pagar = CPV / Fornecedores
df['IND_giro_cp']       = safe_div(df['AUX_cpv_abs'],      df['v09_fornecedores'])

# 5.4 Giro Ativo Circulante = Receita / AC
df['IND_giro_ac']       = safe_div(df['v17_receita_liq'],  df['v02_ativo_circ'])

# 5.5 PMRE = Estoques × 360 / CPV  (NULL para empresas sem estoque)
df['IND_pmre']          = safe_div(df['v06_estoques_raw'] * 360, df['AUX_cpv_abs'])

# 5.6 PMRV = Clientes × 360 / Receita
df['IND_pmrv']          = safe_div(df['v07_contas_receber'] * 360, df['v17_receita_liq'])

# 5.7 PMPC = Fornecedores × 360 / CPV
df['IND_pmpc']          = safe_div(df['v09_fornecedores'] * 360, df['AUX_cpv_abs'])

# 5.8 PMRAC = AC × 360 / Receita
df['IND_pmrac']         = safe_div(df['v02_ativo_circ'] * 360, df['v17_receita_liq'])

print('✅ Giros e Prazos OK')


🔄 Calculando indicadores de Atividade...
✅ Giros e Prazos OK


### 9.6 Ciclos (Seção 6) e Recursos Financeiros / Fleuriet (Seção 7)

In [14]:
print('🔁 Calculando Ciclos e Capital de Giro (Fleuriet)...')

# 6.1 Ciclo Econômico = PMRE + PMRV
# NULL quando PMRE = NULL (empresa sem estoque) → coerente com N/A de PMRE
df['IND_ciclo_economico']  = df['IND_pmre'] + df['IND_pmrv']

# 6.2 Ciclo Financeiro = PMRE + PMRV - PMPC
df['IND_ciclo_financeiro'] = df['IND_pmre'] + df['IND_pmrv'] - df['IND_pmpc']

# 7.1 CGL = AC - PC  (Capital de Giro Líquido)
df['IND_cgl']              = df['v02_ativo_circ'] - df['v10_passivo_circ']

# 7.2 NCG = ACO - PCO  (Necessidade de Capital de Giro)
# ACO = Estoques + Clientes CP  |  PCO = Fornecedores
# v06_estoques = COALESCE(raw, 0): empresa sem estoque contribui 0 para ACO (correto)
df['IND_ncg']              = (
    df['v06_estoques'] + df['v07_contas_receber']
) - df['v09_fornecedores']

# 7.3 Saldo de Tesouraria = ACF - PCF  (Modelo Fleuriet)
# ACF = V15 = v22_caixa_bp + v15b_aplic_fin  →  AUX_acf
# PCF = V16 = V13 = v13_emp_cp (empréstimos CP)
df['IND_st']               = df['AUX_acf'] - df['v13_emp_cp']

print('✅ Ciclos e Fleuriet OK')


🔁 Calculando Ciclos e Capital de Giro (Fleuriet)...
✅ Ciclos e Fleuriet OK


## 10 — Inspeção e Qualidade

In [15]:
ind_cols = [c for c in df.columns if c.startswith('IND_')]
total = len(df)

print(f'📋 COBERTURA DOS INDICADORES  (total linhas = {total:,})')
print(f'{"─"*55}')
print(f'{"Indicador":<30}  {"Preenchido":>10}  {"Cobertura%":>10}')
print(f'{"─"*55}')
for col in ind_cols:
    n = df[col].notna().sum()
    pct = n / total * 100
    flag = '✅' if pct >= 95 else ('⚠️ ' if pct >= 70 else '❌')
    print(f'{flag} {col:<28}  {n:>10,}  {pct:>9.1f}%')


📋 COBERTURA DOS INDICADORES  (total linhas = 2,221)
───────────────────────────────────────────────────────
Indicador                       Preenchido  Cobertura%
───────────────────────────────────────────────────────
✅ IND_liquidez_geral                 2,215       99.7%
✅ IND_liquidez_corrente              2,220      100.0%
✅ IND_liquidez_seca                  2,220      100.0%
✅ IND_liquidez_imediata              2,217       99.8%
✅ IND_pct_cp                         2,218       99.9%
✅ IND_pct_at                         2,218       99.9%
✅ IND_garantia_ct                    2,218       99.9%
✅ IND_composicao_endiv               2,218       99.9%
✅ IND_imob_cp                        2,215       99.7%
✅ IND_imob_at                        2,215       99.7%
✅ IND_margem_bruta                   2,213       99.6%
✅ IND_margem_operacional             2,213       99.6%
✅ IND_margem_liquida                 2,213       99.6%
⚠️  IND_lpa_diluido                    1,925       86.7%
✅ IND_roa

In [16]:
# Verificação de sanidade: Liquidez Corrente deve ser > 0 para empresas solventes
print('🔍 Sanidade — Liquidez Corrente negativa (sinal de PL comprometido):')
neg_lc = df[df['IND_liquidez_corrente'] < 0]
print(f'   {len(neg_lc)} linhas com LC < 0 (PC > AC — tecnicamente insolvente no CP)')

print('\n🔍 Sanidade — ROE com PL negativo (interpretação inversa):')
neg_pl = df[df['v12_pl'] < 0]
print(f'   {len(neg_pl)} linhas com PL < 0 — ROE pode parecer positivo com LL negativo')

print('\n🔍 Sanidade — LPA: distribuição básica (sem outlier Marisa)')
print(df['IND_lpa_diluido'].describe(percentiles=[.25,.5,.75,.95,.99]).round(4))


🔍 Sanidade — Liquidez Corrente negativa (sinal de PL comprometido):
   0 linhas com LC < 0 (PC > AC — tecnicamente insolvente no CP)

🔍 Sanidade — ROE com PL negativo (interpretação inversa):
   163 linhas com PL < 0 — ROE pode parecer positivo com LL negativo

🔍 Sanidade — LPA: distribuição básica (sem outlier Marisa)
count    1925.0000
mean       10.0366
std       286.7731
min     -7319.0000
25%         0.0441
50%         0.7300
75%         1.8962
95%         9.9056
99%        83.6786
max      4559.0000
Name: IND_lpa_diluido, dtype: float64


## 11 — Ordenação das Colunas e Preparação do Mart

In [17]:
# Ordem canônica: Dimensões → V__ inputs → AUX__ → IND__
COLS_DIM = ['CNPJ_CIA', 'DT_REFER', 'razao_social', 'setor', 'tp_merc']

COLS_V = [
    # BP
    'v01_ativo_total', 'v02_ativo_circ', 'v03_ativo_ncirc', 'v04_rlp',
    'v05_imobilizado', 'v06_estoques_raw', 'v06_estoques',
    'v07_contas_receber', 'v09_fornecedores',
    'v10_passivo_circ', 'v11_passivo_ncirc', 'v12_pl',
    'v13_emp_cp', 'v14_emp_lp',
    'v22_caixa_bp', 'v15b_aplic_fin',
    # DRE
    'v17_receita_liq', 'v18_cpv', 'v19_lucro_bruto', 'v20_ebit', 'v21_lucro_liq',
    'v25_lpa_basico', 'v26_lpa_diluido_raw', 'v26_lpa_diluido',
    # DFC
    'v23_caixa_dfc',
]

COLS_AUX = [
    'AUX_passivo_total', 'AUX_acf', 'AUX_divida_bruta',
    'AUX_divida_liquida', 'AUX_capital_invest', 'AUX_cpv_abs',
]

COLS_IND = [
    # Liquidez
    'IND_liquidez_geral', 'IND_liquidez_corrente', 'IND_liquidez_seca', 'IND_liquidez_imediata',
    # Endividamento
    'IND_pct_cp', 'IND_pct_at', 'IND_garantia_ct', 'IND_composicao_endiv',
    'IND_imob_cp', 'IND_imob_at',
    # Margem
    'IND_margem_bruta', 'IND_margem_operacional', 'IND_margem_liquida', 'IND_lpa_diluido',
    # Rentabilidade
    'IND_roa', 'IND_roe', 'IND_roi',
    # Atividade
    'IND_giro_estoques', 'IND_giro_cr', 'IND_giro_cp', 'IND_giro_ac',
    # Prazos
    'IND_pmre', 'IND_pmrv', 'IND_pmpc', 'IND_pmrac',
    # Ciclos
    'IND_ciclo_economico', 'IND_ciclo_financeiro',
    # Capital de Giro / Fleuriet
    'IND_cgl', 'IND_ncg', 'IND_st',
]

df_mart = df[COLS_DIM + COLS_V + COLS_AUX + COLS_IND].copy()

# Uppercase para consistência com o padrão PostgreSQL do projeto
df_mart.columns = [c.upper() for c in df_mart.columns]

print(f'✅ Mart pronto: {df_mart.shape[0]:,} linhas × {df_mart.shape[1]} colunas')
print(f'   Dimensões : {len(COLS_DIM)}')
print(f'   V__ inputs: {len(COLS_V)}')
print(f'   AUX__     : {len(COLS_AUX)}')
print(f'   IND__     : {len(COLS_IND)}')
df_mart.head(2)


✅ Mart pronto: 2,221 linhas × 66 colunas
   Dimensões : 5
   V__ inputs: 25
   AUX__     : 6
   IND__     : 30


,CNPJ_CIA,DT_REFER,RAZAO_SOCIAL,SETOR,TP_MERC,V01_ATIVO_TOTAL,V02_ATIVO_CIRC,V03_ATIVO_NCIRC,V04_RLP,V05_IMOBILIZADO,...,IND_GIRO_AC,IND_PMRE,IND_PMRV,IND_PMPC,IND_PMRAC,IND_CICLO_ECONOMICO,IND_CICLO_FINANCEIRO,IND_CGL,IND_NCG,IND_ST
0,00.070.698/0001-11,2010-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,BOLSA,2.119934e+09,449960000.0,1.669974e+09,799490000.0,197361000.0,...,4.087741,4.823122,62.860840,80.014363,88.068199,67.683962,-12.330401,-177986000.0,186802000.0,-54941000.0
1,00.070.698/0001-11,2011-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,BOLSA,2.170285e+09,457284000.0,1.713001e+09,820292000.0,193114000.0,...,3.012611,3.065170,93.340002,52.313517,119.497655,96.405172,44.091655,-200481000.0,210847000.0,-60851000.0


## 12 — Escrita na Camada Gold

In [18]:
print(f'💎 ESCREVENDO {GOLD_SCHEMA}.{GOLD_TABLE}...')

with engine.begin() as conn:
    # Garante que o schema existe
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}'))

    df_mart.to_sql(
        name=GOLD_TABLE,
        con=conn,
        schema=GOLD_SCHEMA,
        if_exists='replace',    # recria a tabela a cada execução
        index=False,
        method='multi',
        chunksize=500
    )

print(f'✅ {GOLD_SCHEMA}.{GOLD_TABLE} gravado com sucesso!')
print(f'   Linhas: {len(df_mart):,}')
print(f'   Colunas: {len(df_mart.columns)}')


💎 ESCREVENDO layer_03_gold.mart_indicadores_financeiros...
✅ layer_03_gold.mart_indicadores_financeiros gravado com sucesso!
   Linhas: 2,221
   Colunas: 66


In [19]:
# Validação pós-escrita: conta as linhas que voltam do banco
Q_VALIDA = f'SELECT COUNT(*) AS n FROM {GOLD_SCHEMA}.{GOLD_TABLE}'
with engine.connect() as conn:
    n_gold = pd.read_sql(text(Q_VALIDA), conn)['n'].iloc[0]

assert n_gold == len(df_mart), f'ERRO: {n_gold} no banco vs {len(df_mart)} esperado'
print(f'✅ Validação pós-escrita: {n_gold:,} linhas confirmadas no banco.')


✅ Validação pós-escrita: 2,221 linhas confirmadas no banco.
